# tcpyVPI: ERA5 Monthly and Hourly GPIv Computation and Anomalies

This notebook demonstrates how to use the tcpyVPI package to compute
ventilated Potential Intensity (vPI) and Genesis Potential Index (GPIv)
from ERA5 monthly and hourly reanalysis data using the built-in functions specific to ERA5.

**Key Features:**
- Load ERA5 monthly or hourly data via THREDDS
- Compute GPIv at specific times
- Calculate climatologies
- Compute anomalies relative to climatology

**Author:** Jose Ocegueda Sanchez(2025)

## Summary

The tcpyVPI package now supports:

1. **Monthly mean data**: `run_vpigpiv(year, month, data_source='monthly')`
2. **Hourly data**: `run_vpigpiv_hourly(year, month, day, hour=hour)`
3. **Climatology**: `compute_monthly_climatology(...)`
4. **Anomalies**: `compute_anomalies(...)` or `run_vpigpiv_hourly(..., compute_anomalies=True)`

In [3]:
!pip install cartopy tcpyPI tcpyVPI

In [2]:
# Import required packages
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Import tcvpigpiv
from tcpyVPI import (
    run_vpigpiv,
    run_vpigpiv_hourly,
    load_era5_data,
    compute_gpiv_from_dataset,
)

## 1. Monthly Mean GPIv

First, let's compute GPIv from ERA5 monthly mean data (the original functionality).

In [ ]:
# Compute monthly mean GPIv for September 2022
year = 2022
month = 9

results_monthly = run_vpigpiv(year, month, data_source='monthly', plot=True)
print(f"\nVariables computed: {list(results_monthly.data_vars)}")

Loading ERA5 monthly mean data for 2022-09...
  Loading SSTK...
  Loading SP...
  Loading T...
  Loading Q...
  Loading U...
  Loading V...
  Loading VO...


/usr/local/lib/python3.12/dist-packages/tcpyVPI/era5_loader.py:212: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds = xr.merge([


  Data loading complete.
  Calculating Potential Intensity (PI)...
  Calculating Vertical Wind Shear (VWS)...
  Calculating Entropy Deficit (Chi)...
T600: min = 216.72251892089844 , max = 282.0955810546875
rv600: min = 9.35473963181721e-06 , max = 0.008615713566541672
sm_600: min = -85.8638635496223 , max = 255.2927700887706
sm_star_600: min = -85.72642046249604 , max = 291.6622730566628
  Calculating Capped Vorticity (eta_c)...


/usr/local/lib/python3.12/dist-packages/xarray/computation/apply_ufunc.py:818: RuntimeWarning: divide by zero encountered in divide
  result_data = func(*input_data)


  Combining components for final GPIv...
GPIv: min=2.517e-22, max=0.1114, mean=0.009068
vPI: min=2.296, max=130, mean=80.02
PI: min=0, max=131.7, mean=45.57
ventilation_index: min=8.422e-05, max=inf, mean=inf
VWS: min=0.0005493, max=54.79, mean=15.18
Chi: min=-inf, max=inf, mean=nan
eta_c: min=-3.7e-05, max=3.7e-05, mean=3.029e-05


## 2. Hourly GPIv Computation

Now let's compute GPIv from hourly ERA5 data. This allows us to capture
instantaneous atmospheric conditions.

In [ ]:
# Compute hourly GPIv for August 15, 2020 at 12Z
year = 2020
month = 8
day = 15
hour = 12

results_hourly = run_vpigpiv_hourly(year, month, day, hour=hour, plot=True)
print(f"\nVariables computed: {list(results_hourly.data_vars)}")

## 3. Data Loading Details

Understanding how the ERA5 data is organized is important.

**Hourly Dataset (d633000):**
- Surface variables: Monthly files containing all hours
- Pressure level variables: Daily files containing 24 hours

In [ ]:
# Direct data loading example
from tcvpigpiv import load_era5_hourly

# Load hourly data directly
ds = load_era5_hourly(2020, 8, 15, hour=12, verbose=True)

print(f"\nDataset variables: {list(ds.data_vars)}")
print(f"Dataset dimensions: {dict(ds.dims)}")

## 4. Computing Climatology

To compute anomalies, we first need a climatology. Here's how to compute one:

In [ ]:
from tcvpigpiv import compute_monthly_climatology

# Compute a climatology (this takes a while - use fewer years for testing)
# For a full climatology, use years=range(1980, 2020)
climatology = compute_monthly_climatology(
    compute_gpiv_from_dataset,
    months=[8],  # Just August for this example
    years=range(2015, 2020),  # 5 years for quick testing, default is 1980-2020
    data_source='monthly',
    output_path='gpiv_climatology_august.nc',
    verbose=True
)

print(f"\nClimatology variables: {list(climatology.data_vars)}")

## 5. Computing Anomalies

Now we can compute anomalies relative to the climatology:

In [ ]:
from tcvpigpiv import load_climatology, compute_anomalies

# Load the climatology
clim = load_climatology('gpiv_climatology_august.nc')

# Compute anomalies for the hourly data
anomalies = compute_anomalies(results_hourly, clim, month=8)

print(f"\nAnomaly variables: {list(anomalies.data_vars)}")

## 6. Visualization

In [ ]:
# Plot GPIv and its anomaly
fig, axes = plt.subplots(1, 2, figsize=(14, 5),
                         subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180)})

# GPIv
ax = axes[0]
results_hourly['GPIv'].plot(ax=ax, transform=ccrs.PlateCarree(),
                            cmap='viridis', cbar_kwargs={'label': 'GPIv'})
ax.coastlines()
ax.add_feature(cfeature.BORDERS, linestyle=':')
ax.set_title(f'GPIv - {year}-{month:02d}-{day:02d} {hour:02d}Z')
gl = ax.gridlines(draw_labels=True)
gl.top_labels = gl.right_labels = False

# GPIv Anomaly
ax = axes[1]
if 'GPIv_anom' in anomalies:
    anomalies['GPIv_anom'].plot(ax=ax, transform=ccrs.PlateCarree(),
                                cmap='RdBu_r', cbar_kwargs={'label': 'GPIv Anomaly'})
    ax.coastlines()
    ax.add_feature(cfeature.BORDERS, linestyle=':')
    ax.set_title(f'GPIv Anomaly - {year}-{month:02d}-{day:02d} {hour:02d}Z')
    gl = ax.gridlines(draw_labels=True)
    gl.top_labels = gl.right_labels = False

plt.tight_layout()
plt.show()

## 7. Using run_vpigpiv_hourly with Anomalies

The easiest way to compute both GPIv and anomalies in one call:

In [ ]:
# All-in-one computation with anomalies
results_with_anom = run_vpigpiv_hourly(
    2020, 8, 15, hour=12,
    compute_anomalies=True,
    climatology_path='gpiv_climatology_august.nc',
    plot=False
)

print(f"Variables: {list(results_with_anom.data_vars)}")

## 8. Time Series Analysis

Load multiple hours to analyze temporal evolution:

In [ ]:
from tcvpigpiv import load_era5_hourly_range
from datetime import datetime

# Load a range of dates (at specific synoptic hours)
# Warning: This can take a while!
ds_range = load_era5_hourly_range(
    datetime(2020, 8, 15),
    datetime(2020, 8, 16),
    hours=[0, 12],  # Just 00Z and 12Z
    verbose=True
)

print(f"\nTime range: {ds_range.time.values}")